In [ ]:


export=False

from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np



FILE_NEIGHBORHOODS = Path(r'I:\Projects\Jin Eui\Neighborhood_Assoc\Export_Output.shp')
FILE_BLOCK_GROUPS = Path(r'I:\Projects\Josh\Geospatial Data\TIGER\geojson\tl_2020_sacog_bg.geojson')
FILE_ACS = Path(r'C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Products\Small Data Requests\2025\Sac Observer\ACS')



def remove_post(x, exp='.'):
    try: x = x.split(exp, 1)[0]
    except: pass
    return x


def clean_fips(df):
    
    if 'Block Group ID' in df.columns:                   df['Block Group ID'                  ] = df['Block Group ID'                  ].astype(str)
    if 'Tract ID' in df.columns:                         df['Tract ID'                        ] = df['Tract ID'                        ].astype(str).apply('{:0>6}'.format)
    if 'STATEFP' in df.columns:                          df['STATEFP'                         ] = df['STATEFP'                         ].astype(str).apply('{:0>2}'.format)
    if 'State FIPS' in df.columns:                       df['State FIPS'                      ] = df['State FIPS'                      ].astype(str).apply('{:0>2}'.format)
    if 'Place ID' in df.columns:                         df['Place ID'                        ] = df['Place ID'                        ].astype(str).apply('{:0>5}'.format)
    if 'COUNTYFP' in df.columns:                         df['COUNTYFP'                        ] = df['COUNTYFP'                        ].astype(str).apply('{:0>3}'.format)
    if 'County FIPS' in df.columns:                      df['County FIPS'                     ] = df['County FIPS'                     ].astype(str).apply('{:0>3}'.format)
    if 'Congressional District' in df.columns:           df['Congressional District'          ] = df['Congressional District'          ].astype(str).apply('{:0>2}'.format)
    if 'State Legislative Upper District' in df.columns: df['State Legislative Upper District'] = df['State Legislative Upper District'].astype(str).apply('{:0>3}'.format)
    if 'State Legislative Lower District' in df.columns: df['State Legislative Lower District'] = df['State Legislative Lower District'].astype(str).apply('{:0>3}'.format)

    if 'Block Group ID' in df.columns:
        df = df.dropna(subset=['Block Group ID'])
        df = df[df['Block Group ID'] != 'nan']
        df['Block Group ID'] = df['Block Group ID'].apply(remove_post)

    if 'GEOID' not in df.columns:
        if 'State FIPS' in df.columns and 'County FIPS' in df.columns and 'Tract ID' in df.columns and 'Block Group ID' in df.columns:
            df['GEOID'] = df['State FIPS'] + df['County FIPS'] + df['Tract ID'] + df['Block Group ID']
            df['GEOID'] = df['GEOID'].astype('int64')
            df = df.set_index('GEOID').reset_index()

    return df



In [ ]:


gdf_neighborhoods = gpd.read_file(FILE_NEIGHBORHOODS)
display(gdf_neighborhoods)

gdf_bg = gpd.read_file(FILE_BLOCK_GROUPS)
display(gdf_bg.head())


df_pop   = pd.read_excel(FILE_ACS / 'Pop_3 Block Groups ACS5.xlsx'   , sheet_name='Block Groups')
df_owner = pd.read_excel(FILE_ACS / 'Cost_5 Block Groups ACS5.xlsx'  , sheet_name='Block Groups')
df_inc   = pd.read_excel(FILE_ACS / 'Income_1 Block Groups ACS5.xlsx', sheet_name='Block Groups')

df_pop   = df_pop  [df_pop  ['Year'] == 2023].drop('Year', axis=1).dropna().reset_index(drop=True)
df_owner = df_owner[df_owner['Year'] == 2023].drop('Year', axis=1).dropna().reset_index(drop=True)
df_inc   = df_inc  [df_inc  ['Year'] == 2023].drop('Year', axis=1).dropna().reset_index(drop=True)

display(df_pop  .head())
display(df_owner.head())
display(df_inc  .head())



In [ ]:


df_hcost = pd.read_excel(FILE_ACS / 'HousingCost_1 Block Groups ACS5.xlsx', sheet_name='Block Groups')
df_hcost = df_hcost[df_hcost['Year'] == 2023].drop('Year', axis=1).reset_index(drop=True)
display(df_hcost.head())

df_hcost = df_hcost.rename(columns={'Total':'Households'})
df_hcost = df_hcost[~df_hcost['Block Group ID'].isna()]
df_hcost = df_hcost.fillna(0)

df_hcost   = clean_fips(df_hcost)

df_hcost.loc[df_hcost['Variable'].str.contains('with a mortgage'   ), 'Type'] = 'Owner with a mortgage'
df_hcost.loc[df_hcost['Variable'].str.contains('without a mortgage'), 'Type'] = 'Owner without a mortgage'
df_hcost.loc[df_hcost['Variable'].str.contains('Rented'            ), 'Type'] = 'Renter'

df_hcost = df_hcost[['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Variable', 'Households', 'Margin of Error']]

df_total = df_hcost.copy()
df_total = df_total[df_total['Variable'].isin(['Owned units with a mortgage', 'Owned units without a mortgage', 'Rented units'])].reset_index(drop=True)
df_total = df_total[['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Households', 'Margin of Error']].rename(columns={'Margin of Error':'Margin of Error Total'})
df_total = df_total.rename(columns={'Households':'Total Households'})

df_total_bg = df_total[df_total['Race_Ethnicity']=='All']
df_total_bg = df_total_bg.groupby(['GEOID'], as_index=False)['Total Households'].sum()

df_sub = df_hcost.copy()
df_sub = df_sub[~df_sub['Variable'].isin(['Owned units with a mortgage', 'Owned units without a mortgage', 'Rented units'])]

sqrtsumsq  = lambda x: np.sqrt(np.sum(x**2)) # Square root of the sum of squares (to roll up SE's when +/- random variables)
df_sub = df_sub.groupby(['GEOID', 'NAME', 'Race_Ethnicity', 'Type'], as_index=False).agg(Households=('Households', 'sum'), ME=('Margin of Error', sqrtsumsq))
df_sub = df_sub[['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Households', 'ME']].drop_duplicates().reset_index(drop=True)
df_sub = df_sub.rename(columns={'Households':'Households over 30 pct or not computed'})

df_under30 = df_total.merge(df_sub, on=['GEOID', 'NAME', 'Race_Ethnicity', 'Type'], how='left')
df_under30['Households'] = df_under30['Total Households'] - df_under30['Households over 30 pct or not computed']
df_under30.loc[df_under30['Households']<0, 'Households'] = 0
df_under30['Margin of Error'] = np.sqrt(df_under30['Margin of Error Total']**2 + df_under30['ME']**2)
df_under30.loc[df_under30['Type'] == 'Owner with a mortgage'   , 'Variable'] = 'Owned units with a mortgage under 30 pct'
df_under30.loc[df_under30['Type'] == 'Owner without a mortgage', 'Variable'] = 'Owned units without a mortgage under 30 pct'
df_under30.loc[df_under30['Type'] == 'Renter'                  , 'Variable'] = 'Rented units under 30 pct'
df_under30 = df_under30[['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Variable', 'Households', 'Margin of Error']]
display(df_under30.head())

df_hcost = pd.concat([df_hcost, df_under30])
df_hcost = df_hcost[~df_hcost['Variable'].isin(['Owned units with a mortgage', 'Owned units without a mortgage', 'Rented units'])]
df_hcost = df_hcost.sort_values(['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Variable'])
df_hcost = df_hcost.reset_index(drop=True)
df_hcost['Percentage'] = df_hcost['Households'] / df_hcost.groupby(['GEOID', 'NAME', 'Race_Ethnicity', 'Type'])['Households'].transform('sum')

df_hcost.head(10)



In [ ]:


## Merge ACS data onto intersected block group shapefile using GEOID
df_pop   = clean_fips(df_pop  )
df_owner = clean_fips(df_owner)
df_inc   = clean_fips(df_inc  )

df_pop   = df_pop  [['GEOID', 'NAME', 'Race_Ethnicity', 'Population', 'Margin of Error']]
df_owner = df_owner[['GEOID', 'NAME', 'Race_Ethnicity', 'Variable', 'Households', 'Margin of Error']]
df_inc   = df_inc  [['GEOID', 'NAME', 'Median Household Income', 'Margin of Error']]
df_hcost = df_hcost[['GEOID', 'NAME', 'Race_Ethnicity', 'Type', 'Variable', 'Households', 'Margin of Error']]

display(df_pop  .head())
display(df_owner.head())
display(df_inc  .head())
display(df_hcost.head())



In [ ]:


## Intersect block group level shapefile with neighborhoods shapefile
gdf_bg_neighborhooods = gpd.overlay(gdf_neighborhoods, gdf_bg)
df_bg_neighborhooods = gdf_bg_neighborhooods[['GEOID', 'NEW_NAME']].drop_duplicates().reset_index(drop=True)

gdf_bg = gdf_bg[gdf_bg['GEOID'].isin(df_bg_neighborhooods['GEOID'].unique())].reset_index(drop=True)
gdf_bg_neighborhoods = gdf_bg.merge(df_bg_neighborhooods, on='GEOID', how='left')
gdf_bg_neighborhoods = gdf_bg_neighborhoods[['NEW_NAME', 'GEOID', 'geometry']].sort_values(['NEW_NAME', 'GEOID']).reset_index(drop=True)
display(gdf_bg_neighborhoods.head())


gdf_bg_nhds_pop   = gdf_bg_neighborhoods.merge(df_pop  , on='GEOID', how='left').reset_index(drop=True)
gdf_bg_nhds_own   = gdf_bg_neighborhoods.merge(df_owner, on='GEOID', how='left').reset_index(drop=True)
gdf_bg_nhds_inc   = gdf_bg_neighborhoods.merge(df_inc  , on='GEOID', how='left').reset_index(drop=True)
gdf_bg_nhds_hcost = gdf_bg_neighborhoods.merge(df_hcost, on='GEOID', how='left').reset_index(drop=True)

df_bg_nhds_pop   = gdf_bg_nhds_pop  .drop('geometry', axis=1)
df_bg_nhds_own   = gdf_bg_nhds_own  .drop('geometry', axis=1)
df_bg_nhds_inc   = gdf_bg_nhds_inc  .drop('geometry', axis=1)
df_bg_nhds_hcost = gdf_bg_nhds_hcost.drop('geometry', axis=1)


sqrtsumsq  = lambda x: np.sqrt(np.sum(x**2)) # Square root of the sum of squares (to roll up SE's when +/- random variables)
sumsq  = lambda x: np.sum(x**2)
nsq = lambda x: len(x)**2

df_bg_nhds_pop.loc[df_bg_nhds_pop['Margin of Error'] < 0, 'Margin of Error'] = np.nan
df_bg_nhds_own.loc[df_bg_nhds_own['Margin of Error'] < 0, 'Margin of Error'] = np.nan
df_bg_nhds_inc.loc[df_bg_nhds_inc['Margin of Error'] < 0, 'Margin of Error'] = np.nan
df_bg_nhds_hcost.loc[df_bg_nhds_hcost['Margin of Error'] < 0, 'Margin of Error'] = np.nan

df_nhds_pop = df_bg_nhds_pop.groupby(['NEW_NAME', 'Race_Ethnicity'            ], as_index=False).agg(Population=('Population', 'sum'), ME=('Margin of Error', sqrtsumsq)).rename(columns={'ME':'Margin of Error'})
df_nhds_own = df_bg_nhds_own.groupby(['NEW_NAME', 'Race_Ethnicity', 'Variable'], as_index=False).agg(Households=('Households', 'sum'), ME=('Margin of Error', sqrtsumsq)).rename(columns={'ME':'Margin of Error'})

df_nhds_inc = gdf_bg_nhds_inc.dropna().merge(df_total_bg, on=['GEOID'], how='left').reset_index(drop=True)
wm = lambda x: np.average(x, weights = df_nhds_inc.loc[x.index, 'Total Households'])
df_nhds_inc = df_nhds_inc.groupby(['NEW_NAME'], as_index=False).agg(Income=('Median Household Income', wm), ME=('Margin of Error', sumsq), nsq=('GEOID', nsq)).rename(columns={'Income':'Median Household Income', 'ME':'Margin of Error'})
df_nhds_inc['Margin of Error'] = np.sqrt(df_nhds_inc['Margin of Error']/df_nhds_inc['nsq'])

df_nhds_hcost = gdf_bg_nhds_hcost.drop('geometry', axis=1).groupby(['NEW_NAME', 'Race_Ethnicity', 'Type', 'Variable'], as_index=False).agg(Households=('Households', 'sum'), ME=('Margin of Error', sqrtsumsq)).rename(columns={'ME':'Margin of Error'})

df_nhds_pop  ['Percentage'] = df_nhds_pop  ['Population'] / df_nhds_pop  .groupby(['NEW_NAME'                          ])['Population'].transform('sum')
df_nhds_own  ['Percentage'] = df_nhds_own  ['Households'] / df_nhds_own  .groupby(['NEW_NAME', 'Race_Ethnicity'        ])['Households'].transform('sum')
df_nhds_hcost['Percentage'] = df_nhds_hcost['Households'] / df_nhds_hcost.groupby(['NEW_NAME', 'Race_Ethnicity', 'Type'])['Households'].transform('sum')

df_nhds_pop   = df_nhds_pop  [['NEW_NAME', 'Race_Ethnicity',                     'Population',               'Percentage', 'Margin of Error']].rename(columns={'NEW_NAME':'Neighborhood'})
df_nhds_own   = df_nhds_own  [['NEW_NAME', 'Race_Ethnicity', 'Variable',         'Households',               'Percentage', 'Margin of Error']].rename(columns={'NEW_NAME':'Neighborhood', 'Variable':'Household Type'})
df_nhds_inc   = df_nhds_inc  [['NEW_NAME',                                       'Median Household Income',                'Margin of Error']].rename(columns={'NEW_NAME':'Neighborhood'})
df_nhds_hcost = df_nhds_hcost[['NEW_NAME', 'Race_Ethnicity', 'Type', 'Variable', 'Households',               'Percentage', 'Margin of Error']].rename(columns={'NEW_NAME':'Neighborhood', 'Type':'Household Type', 'Variable':'Cost Burden'})


display(df_nhds_pop  .head(10))
display(df_nhds_own  .head(10))
display(df_nhds_inc  .head(10))
display(df_nhds_hcost.head(10))




In [ ]:


export=False

if export:
    with pd.ExcelWriter(FILE_ACS / 'ACS Supplement copy.xlsx', engine='xlsxwriter') as writer:
        df_nhds_pop  .to_excel(writer, index=False, sheet_name='Population'   )
        df_nhds_inc  .to_excel(writer, index=False, sheet_name='Income'       )
        df_nhds_own  .to_excel(writer, index=False, sheet_name='HomeOwnership')
        df_nhds_hcost.to_excel(writer, index=False, sheet_name='CostBurden'   )

